# Bảng Điều Khiển Tương Tác Thị Trường Xe Cũ

Mục đích:
Lọc dữ liệu theo Hãng xe, Dòng xe, Tỉnh/Thành và Khoảng năm sản xuất. Khi thay đổi bộ lọc, biểu đồ phân phối giá và tương quan ODO - Giá bán sẽ tự động cập nhật ngay lập tức.

In [1]:
from pathlib import Path
import ipywidgets as widgets
import pandas as pd
import plotly.express as px
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "requirements.txt").exists())
DATA_PATH = ROOT / "data" / "processed" / "cars_cleaned.csv"
df = pd.read_csv(DATA_PATH, low_memory=False)

print(f"Đã nạp dữ liệu: {df.shape[0]:,} xe")

Đã nạp dữ liệu: 13,520 xe


In [2]:
def update_dashboard(brand, model, province, years):
    sub = df.copy()
    if brand != "Tất cả":
        sub = sub[sub["carbrand_name"] == brand]
    if model != "Tất cả":
        sub = sub[sub["carmodel_name"] == model]
    if province != "Tất cả":
        sub = sub[sub["region_name"] == province]
    if years:
        sub = sub[(sub["mfdate"] >= years[0]) & (sub["mfdate"] <= years[1])]

    if sub.empty:
        print("Không có xe phù hợp với bộ lọc hiện tại.")
        return

    med_p = sub["price_million"].median()
    print(f"Tổng số: {len(sub):,} xe | Giá trung vị: {med_p:.1f} triệu | ODO trung vị: {sub['mileage_v2'].median():,.0f} km")

    px.histogram(sub, x="price_million", nbins=30, title=f"Phân Phối Giá Xe (Trung vị: {med_p:.1f} Triệu VNĐ)").show()
    sub["odo_group"] = pd.cut(
        sub["mileage_v2"],
        bins=[0, 20000, 50000, 100000, float("inf")],
        labels=["Dưới 20k km", "20k - 50k km", "50k - 100k km", "Trên 100k km"],
        right=False,
    )
    px.box(
        sub,
        x="odo_group",
        y="price_million",
        title="Tương Quan ODO và Giá Bán",
        points=False,
        labels={"odo_group": "Khoảng ODO", "price_million": "Giá (Triệu VNĐ)"},
    ).show()

In [3]:
w_brand = widgets.Dropdown(options=["Tất cả"] + sorted(df["carbrand_name"].dropna().unique()), description="Hãng xe:")
w_model = widgets.Dropdown(options=["Tất cả"], description="Dòng xe:")
w_province = widgets.Dropdown(options=["Tất cả"] + sorted(df["region_name"].dropna().unique()), description="Tỉnh/Thành:")
w_year = widgets.IntRangeSlider(
    min=int(df["mfdate"].min()),
    max=int(df["mfdate"].max()),
    value=[2015, int(df["mfdate"].max())],
    description="Năm SX:",
)

def on_brand_change(change):
    brand = change.new
    models = sorted(df[df["carbrand_name"] == brand]["carmodel_name"].dropna().unique()) if brand != "Tất cả" else []
    w_model.options = ["Tất cả"] + models
    w_model.value = "Tất cả"

w_brand.observe(on_brand_change, names="value")

out = widgets.interactive_output(
    update_dashboard,
    {"brand": w_brand, "model": w_model, "province": w_province, "years": w_year},
)

display(widgets.VBox([
    widgets.HBox([w_brand, w_model]),
    widgets.HBox([w_province, w_year]),
    out,
]))